# Experiment 2 — multi-seed replication of the Qwen2.5-VL-7B QLoRA winner

This notebook replicates the repository's held-out-test winner: the **2,600-example,
510-step, frozen-vision QLoRA recipe**. It is derived from
`Development/qwen2p5-3b-7b/qlora-q7b-2k-image/step1-qlora-upto-200-chkpoint.ipynb`
and incorporates the documented 2.6k winner settings from `README.md`.

Required seeds: **13, 42, 73**. Optional seeds: **101, 202**.

The seed controls the training-subset permutation, DataLoader/epoch order, LoRA
initialization, dropout RNG, CUDA/PyTorch RNG, and preprocessing-worker RNG. Qwen's
current resize/normalization path is deterministic; worker seeding also protects the
experiment if stochastic image transforms are introduced later.

Outputs per seed include an adapter, checkpoints, an auditable manifest, training log,
and row-level dev predictions. The final cells report every seed's CI/accuracy, mean,
sample standard deviation, Wilson intervals, pairwise exact McNemar tests, paired
bootstrap confidence intervals, a majority-vote ensemble, and a seed medoid.

> This notebook is intentionally unexecuted. Run on a GPU Kaggle runtime. Three 7B
> trainings may exceed a single session; in that case set `SEEDS_TO_RUN` to one seed,
> preserve that run's output, and list attached run roots in `EXTERNAL_RESULT_ROOTS`
> before running the aggregation cells.


## 1. Environment


In [ ]:
import os, warnings

# Set deterministic CUDA environment flags before importing torch.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['BITSANDBYTES_NOWELCOME'] = '1'
os.environ['BNB_CUDA_VERSION'] = '128'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore', message='.*use_reentrant.*')

for _major in ('12', '13'):
    _src = f'/usr/local/cuda/lib64/libnvJitLink.so.{_major}'
    _dst = '/usr/local/cuda/lib64/libnvJitLink.so.13'
    if os.path.exists(_src) and not os.path.exists(_dst):
        os.symlink(_src, _dst)
        print(f'Symlinked .{_major} -> .13')
        break

!pip install -q -U 'transformers>=4.49.0' 'peft>=0.10.0' accelerate bitsandbytes qwen-vl-utils scipy 2>&1 | tail -4
print('Dependencies ready.')


## 2. Pre-registered configuration


In [ ]:
from pathlib import Path
import gc, hashlib, json, math, random, re, time
import numpy as np
import pandas as pd
import torch
from transformers import set_seed

REPO_ID = 'QCRI/AynVQA-ArabicNLP26'
TASK = 'task1b'
LANG = 'en'
VLM_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'

# Held-out-test winner documented in README.md.
RECIPE_NAME = 'qwen2p5-vl-7b_qlora_frozen-vision_2p6k_510steps'
TRAIN_SUBSAMPLE_N = 2600
MAX_STEPS = 510
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
MAX_SEQ_LEN = 1280
TRAIN_MAX_PIXELS = 256 * 28 * 28
EVAL_MAX_PIXELS = 1024 * 28 * 28
MAX_NEW_TOKENS = 256

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGETS = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

REQUIRED_SEEDS = [13, 42, 73]
OPTIONAL_SEEDS = [101, 202]
RUN_OPTIONAL_SEEDS = False
SEEDS_TO_RUN = REQUIRED_SEEDS + (OPTIONAL_SEEDS if RUN_OPTIONAL_SEEDS else [])

SAVE_STEPS = 200
LOGGING_STEPS = 10
NUM_WORKERS = 2
OUTPUT_ROOT = Path('/kaggle/working/experiment2_seed_replication')

# Add attached Kaggle output roots here when seeds were trained in separate sessions.
# Each root must contain seed_13/, seed_42/, ... with predictions_dev.csv files.
EXTERNAL_RESULT_ROOTS = []
BOOTSTRAP_REPLICATES = 20_000
BOOTSTRAP_SEED = 20260809

assert len(set(SEEDS_TO_RUN)) == len(SEEDS_TO_RUN)
assert set(REQUIRED_SEEDS).issubset(set(REQUIRED_SEEDS + OPTIONAL_SEEDS))
assert TRAIN_SUBSAMPLE_N <= 3000
assert MAX_STEPS <= math.ceil(TRAIN_SUBSAMPLE_N * NUM_EPOCHS / GRAD_ACCUM)
assert torch.cuda.is_available(), 'A CUDA GPU is required.'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Recipe:', RECIPE_NAME)
print('Seeds scheduled:', SEEDS_TO_RUN)
print('Output:', OUTPUT_ROOT)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} ({p.total_memory / 1024**3:.1f} GiB)')


## 3. Determinism contract

Seed derivations are explicit so that subset selection, epoch shuffling, and worker RNG
are reproducible but not accidentally coupled. `warn_only=True` records unsupported
nondeterministic CUDA operations without crashing a long run. Reproducibility still
requires the same package, driver, CUDA, GPU architecture, and model revisions; those
are captured in each manifest.


In [ ]:
def seed_everything(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int):
    # PyTorch assigns each worker a deterministic initial seed from loader_generator.
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


def make_generator(seed: int):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def stable_hash(values):
    payload = json.dumps(values, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def package_versions():
    import platform, transformers, peft, bitsandbytes
    return {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'peft': peft.__version__,
        'bitsandbytes': bitsandbytes.__version__,
        'cuda': torch.version.cuda,
        'cudnn': str(torch.backends.cudnn.version()),
        'gpu': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    }


seed_everything(REQUIRED_SEEDS[0])
print('Deterministic controls enabled.')


## 4. Download train/dev metadata and the union of required images


In [ ]:
from huggingface_hub import hf_hub_download, login
from tqdm.auto import tqdm

# Never embed a token in a notebook. Public downloads need no login; set HF_TOKEN in
# Kaggle Secrets only if authentication is required.
if os.getenv('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)


def read_split(split):
    path = hf_hub_download(
        REPO_ID, filename=f'{TASK}/{split}_{LANG}.jsonl', repo_type='dataset')
    with open(path, encoding='utf-8') as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    assert rows and 'labels' in rows[0], f'{split} must be labelled for this experiment.'
    return rows


all_train_records = read_split('train')
dev_records = read_split('dev')
assert TRAIN_SUBSAMPLE_N <= len(all_train_records)
print(f'Train={len(all_train_records)} | dev={len(dev_records)}')


def seeded_subset(records, seed, n):
    # A full permutation followed by a prefix makes the selection rule auditable.
    order = list(range(len(records)))
    random.Random(seed).shuffle(order)
    return [records[i] for i in order[:n]], order


seed_subsets = {}
seed_permutations = {}
for seed in SEEDS_TO_RUN:
    subset, permutation = seeded_subset(all_train_records, seed, TRAIN_SUBSAMPLE_N)
    seed_subsets[seed] = subset
    seed_permutations[seed] = permutation
    print(seed, len(subset), stable_hash([r['id'] for r in subset])[:16])

needed_images = {r['image'] for r in dev_records}
for subset in seed_subsets.values():
    needed_images.update(r['image'] for r in subset)

image_paths, failed_images = {}, []
for rel in tqdm(sorted(needed_images), desc='images'):
    try:
        image_paths[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type='dataset')
    except Exception as exc:
        failed_images.append({'image': rel, 'error': repr(exc)})

assert not failed_images, f'Image downloads failed: {failed_images[:5]}'
print(f'Downloaded {len(image_paths)}/{len(needed_images)} images.')


## 5. Dataset and fixed prompt


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

SYSTEM_PROMPT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
    'Below are THREE statements about this image. '
    'Exactly ONE statement is grounded in the image (True). '
    'The other two are plausible-sounding hallucinations (False).'
)

USER_TEMPLATE = (
    'Statement 1: {s0}\n'
    'Statement 2: {s1}\n'
    'Statement 3: {s2}\n\n'
    'Instructions:\n'
    '- On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.\n'
    '- For each statement evaluate:\n'
    '    (a) Colour/texture evidence for or against\n'
    '    (b) Shape/form evidence for or against\n'
    '    (c) Contextual evidence for or against\n'
    '- Then state your conclusion.\n'
    'Do not write anything before the Answer line.'
)


class HalDetectTrainDataset(Dataset):
    def __init__(self, records, paths, processor, max_seq_len):
        self.records = records
        self.paths = paths
        self.processor = processor
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        true_idx = record['labels'].index(True)
        target = f'Answer: {true_idx + 1}'
        user_text = USER_TEMPLATE.format(
            s0=record['statements'][0],
            s1=record['statements'][1],
            s2=record['statements'][2],
        )
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': self.paths[record['image']]},
                {'type': 'text', 'text': user_text},
            ]},
            {'role': 'assistant', 'content': target},
        ]
        prompt_text = self.processor.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True)
        full_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False)
        image_inputs, _ = process_vision_info(messages)
        enc_full = self.processor(
            text=[full_text], images=image_inputs, truncation=False, return_tensors='pt')
        enc_prompt = self.processor(
            text=[prompt_text], images=image_inputs, truncation=False, return_tensors='pt')

        input_ids = enc_full['input_ids'][0][:self.max_seq_len]
        attention_mask = enc_full['attention_mask'][0][:self.max_seq_len]
        prompt_len = min(enc_prompt['input_ids'].shape[1], self.max_seq_len)
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = 0
        pad_len = self.max_seq_len - input_ids.shape[0]
        if pad_len:
            input_ids = torch.cat([
                input_ids, torch.full((pad_len,), pad_id, dtype=torch.long)])
            attention_mask = torch.cat([
                attention_mask, torch.zeros(pad_len, dtype=torch.long)])
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': enc_full['pixel_values'],
            'image_grid_thw': enc_full['image_grid_thw'],
            'labels': labels,
        }


def collate_fn(batch):
    return {
        'input_ids': torch.stack([x['input_ids'] for x in batch]),
        'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch]),
        'pixel_values': torch.cat([x['pixel_values'] for x in batch], dim=0),
        'image_grid_thw': torch.cat([x['image_grid_thw'] for x in batch], dim=0),
    }


train_processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=TRAIN_MAX_PIXELS)
eval_processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=EVAL_MAX_PIXELS)
print('Processors ready.')


## 6. Fresh model and LoRA initialization for each seed


In [ ]:
from collections import Counter
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training


def build_fresh_model(seed):
    # Called before base loading and get_peft_model so LoRA initialization is seeded.
    seed_everything(seed)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )
    max_memory = {i: '13500MiB' for i in range(torch.cuda.device_count())}
    max_memory['cpu'] = '4GiB'
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VLM_MODEL,
        torch_dtype=dtype,
        device_map='auto',
        max_memory=max_memory,
        quantization_config=quant_config,
    )
    base_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias='none',
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(base_model, lora_config)
    for name, parameter in model.named_parameters():
        if 'visual' in name:
            parameter.requires_grad = False

    # Hash initialized LoRA tensors before the first optimizer update.
    digest = hashlib.sha256()
    with torch.no_grad():
        for name, parameter in sorted(model.named_parameters()):
            if parameter.requires_grad:
                digest.update(name.encode('utf-8'))
                digest.update(parameter.detach().float().cpu().numpy().tobytes())
    init_hash = digest.hexdigest()
    print('Layer distribution:', dict(Counter(model.hf_device_map.values())))
    model.print_trainable_parameters()
    return model, dtype, init_hash


## 7. Deterministic training


In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup


def train_one_seed(seed, model, dtype, seed_dir):
    dataset = HalDetectTrainDataset(
        seed_subsets[seed], image_paths, train_processor, MAX_SEQ_LEN)
    loader_seed = seed + 10_000
    train_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=make_generator(loader_seed),
        worker_init_fn=seed_worker,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        prefetch_factor=2 if NUM_WORKERS else None,
        persistent_workers=bool(NUM_WORKERS),
        pin_memory=False,
    )
    available_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM
    assert available_steps >= MAX_STEPS, (available_steps, MAX_STEPS)
    warmup_steps = int(MAX_STEPS * WARMUP_RATIO)
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=MAX_STEPS,
    )
    model.train()
    for module in model.modules():
        if 'Visual' in type(module).__name__:
            module.eval()

    log_rows = []
    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    started = time.time()

    for epoch in range(NUM_EPOCHS):
        if global_step >= MAX_STEPS:
            break
        running_loss = 0.0
        running_batches = 0
        pbar = tqdm(train_loader, desc=f'seed={seed} epoch={epoch + 1}')
        for batch_index, batch in enumerate(pbar):
            batch['pixel_values'] = batch['pixel_values'].to(dtype)
            batch = {
                key: value.to('cuda:0') if torch.is_tensor(value) else value
                for key, value in batch.items()
            }
            outputs = model(**batch)
            unscaled_loss = outputs.loss
            (unscaled_loss / GRAD_ACCUM).backward()
            running_loss += unscaled_loss.item()
            running_batches += 1

            if (batch_index + 1) % GRAD_ACCUM != 0:
                continue
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            row = {
                'seed': seed,
                'epoch': epoch + 1,
                'global_step': global_step,
                'loss': float(unscaled_loss.item()),
                'running_loss': running_loss / running_batches,
                'learning_rate': scheduler.get_last_lr()[0],
                'elapsed_hours': (time.time() - started) / 3600,
            }
            log_rows.append(row)
            pbar.set_postfix(step=global_step, loss=f'{row["running_loss"]:.4f}')

            if global_step % LOGGING_STEPS == 0:
                print(row)
            if global_step % SAVE_STEPS == 0:
                checkpoint_dir = seed_dir / 'checkpoints' / f'step_{global_step}'
                checkpoint_dir.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(checkpoint_dir)
                train_processor.save_pretrained(checkpoint_dir)
            if global_step >= MAX_STEPS:
                break

    assert global_step == MAX_STEPS, f'Expected {MAX_STEPS}, got {global_step}'
    pd.DataFrame(log_rows).to_csv(seed_dir / 'training_log.csv', index=False)
    adapter_dir = seed_dir / 'adapter_final'
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir)
    train_processor.save_pretrained(adapter_dir)
    return log_rows


## 8. Fixed greedy dev evaluation


In [ ]:
@torch.no_grad()
def generate_answer(model, record):
    user_text = USER_TEMPLATE.format(
        s0=record['statements'][0],
        s1=record['statements'][1],
        s2=record['statements'][2],
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image_paths[record['image']]},
            {'type': 'text', 'text': user_text},
        ]},
    ]
    prompt = eval_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = eval_processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to('cuda:0')
    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
    )
    continuation = generated[0][inputs.input_ids.shape[1]:]
    raw = eval_processor.decode(continuation, skip_special_tokens=True).strip()
    del inputs, generated
    return raw


def parse_choice(raw):
    nonempty = [line.strip() for line in raw.splitlines() if line.strip()]
    if nonempty:
        match = re.search(r'answer\s*[:\-]?\s*([123])', nonempty[0], re.I)
        if match:
            return int(match.group(1)) - 1, 'first_line'
    match = re.search(r'answer\s*[:\-]?\s*([123])', raw, re.I)
    if match:
        return int(match.group(1)) - 1, 'later_line'
    return None, 'unparsed'


def evaluate_dev(seed, model, seed_dir):
    # Reset RNG even though decoding is greedy; this protects future changes.
    seed_everything(seed + 20_000)
    model.eval()
    rows = []
    for record in tqdm(dev_records, desc=f'dev seed={seed}'):
        raw = generate_answer(model, record)
        pred_idx, parse_mode = parse_choice(raw)
        gold_idx = record['labels'].index(True)
        rows.append({
            'id': record['id'],
            'gold_idx': gold_idx,
            'pred_idx': pred_idx,
            'correct': pred_idx == gold_idx,
            'parse_mode': parse_mode,
            'raw': raw,
        })
    frame = pd.DataFrame(rows)
    assert frame['id'].is_unique
    assert frame['pred_idx'].notna().all(), (
        'Unparsed dev outputs must be inspected; no arbitrary fallback is allowed.')
    frame['pred_idx'] = frame['pred_idx'].astype(int)
    frame.to_csv(seed_dir / 'predictions_dev.csv', index=False)
    accuracy = frame['correct'].mean()
    print(f'seed={seed}: accuracy={accuracy:.4f}, CI={1 - accuracy:.4f}')
    return frame


## 9. Run the scheduled seeds

Every seed starts from a freshly loaded base model and freshly initialized LoRA adapter.
Completed seeds are skipped if their dev prediction file already exists. This makes a
restarted notebook safe, provided `/kaggle/working` has been restored.


In [ ]:
def run_seed(seed):
    seed_dir = OUTPUT_ROOT / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = seed_dir / 'predictions_dev.csv'
    if prediction_path.exists():
        print(f'Skipping completed seed {seed}: {prediction_path}')
        return

    seed_everything(seed)
    subset_ids = [record['id'] for record in seed_subsets[seed]]
    model, dtype, lora_init_hash = build_fresh_model(seed)
    manifest = {
        'recipe_name': RECIPE_NAME,
        'seed': seed,
        'seed_derivations': {
            'subset_permutation': seed,
            'global_training_lora_dropout': seed,
            'dataloader_epoch_order': seed + 10_000,
            'dev_evaluation': seed + 20_000,
        },
        'train_subsample_n': TRAIN_SUBSAMPLE_N,
        'max_steps': MAX_STEPS,
        'num_epochs': NUM_EPOCHS,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation': GRAD_ACCUM,
        'learning_rate': LEARNING_RATE,
        'warmup_ratio': WARMUP_RATIO,
        'train_max_pixels': TRAIN_MAX_PIXELS,
        'eval_max_pixels': EVAL_MAX_PIXELS,
        'lora': {
            'rank': LORA_RANK,
            'alpha': LORA_ALPHA,
            'dropout': LORA_DROPOUT,
            'targets': LORA_TARGETS,
            'initialization_sha256': lora_init_hash,
        },
        'subset_order_sha256': stable_hash(subset_ids),
        'subset_ids_in_order': subset_ids,
        'packages': package_versions(),
    }
    with open(seed_dir / 'manifest.json', 'w', encoding='utf-8') as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)

    try:
        train_one_seed(seed, model, dtype, seed_dir)
        evaluate_dev(seed, model, seed_dir)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


for experiment_seed in SEEDS_TO_RUN:
    run_seed(experiment_seed)

print('Scheduled runs complete.')


## 10. Load completed runs and report individual/aggregate scores


In [ ]:
def find_prediction_file(seed):
    roots = [OUTPUT_ROOT] + [Path(path) for path in EXTERNAL_RESULT_ROOTS]
    matches = [root / f'seed_{seed}' / 'predictions_dev.csv' for root in roots]
    matches = [path for path in matches if path.exists()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one predictions file for seed {seed}; found {matches}')
    return matches[0]


analysis_seeds = REQUIRED_SEEDS + [
    seed for seed in OPTIONAL_SEEDS
    if any((Path(root) / f'seed_{seed}' / 'predictions_dev.csv').exists()
           for root in [OUTPUT_ROOT] + [Path(x) for x in EXTERNAL_RESULT_ROOTS])
]
prediction_frames = {
    seed: pd.read_csv(find_prediction_file(seed)).sort_values('id').reset_index(drop=True)
    for seed in analysis_seeds
}

reference = prediction_frames[analysis_seeds[0]]
for seed, frame in prediction_frames.items():
    assert frame['id'].tolist() == reference['id'].tolist(), f'ID mismatch: {seed}'
    assert frame['gold_idx'].tolist() == reference['gold_idx'].tolist(), f'Gold mismatch: {seed}'
    assert frame['pred_idx'].notna().all(), f'Unparsed outputs: {seed}'


def wilson_interval(successes, total, z=1.959963984540054):
    p = successes / total
    denominator = 1 + z*z/total
    center = (p + z*z/(2*total)) / denominator
    margin = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
    return center - margin, center + margin


individual_rows = []
for seed, frame in prediction_frames.items():
    n = len(frame)
    errors = int((frame['pred_idx'] != frame['gold_idx']).sum())
    ci = errors / n
    low, high = wilson_interval(errors, n)
    individual_rows.append({
        'seed': seed,
        'n': n,
        'errors': errors,
        'accuracy': 1 - ci,
        'CI': ci,
        'CI_wilson_low': low,
        'CI_wilson_high': high,
    })

individual = pd.DataFrame(individual_rows).sort_values('seed')
aggregate = pd.DataFrame([{
    'n_seeds': len(individual),
    'mean_CI': individual['CI'].mean(),
    'sample_std_CI': individual['CI'].std(ddof=1),
    'mean_accuracy': individual['accuracy'].mean(),
    'sample_std_accuracy': individual['accuracy'].std(ddof=1),
}])
display(individual)
display(aggregate)

REPORT_DIR = OUTPUT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
individual.to_csv(REPORT_DIR / 'individual_seed_scores.csv', index=False)
aggregate.to_csv(REPORT_DIR / 'aggregate_seed_scores.csv', index=False)


## 11. Paired bootstrap and exact McNemar tests on dev predictions


In [ ]:
from itertools import combinations
from scipy.stats import binomtest


def paired_comparison(seed_a, seed_b):
    a = prediction_frames[seed_a]
    b = prediction_frames[seed_b]
    error_a = (a['pred_idx'].to_numpy() != a['gold_idx'].to_numpy()).astype(float)
    error_b = (b['pred_idx'].to_numpy() != b['gold_idx'].to_numpy()).astype(float)
    correct_a = 1 - error_a
    correct_b = 1 - error_b
    a_only_correct = int(((correct_a == 1) & (correct_b == 0)).sum())
    b_only_correct = int(((correct_a == 0) & (correct_b == 1)).sum())
    discordant = a_only_correct + b_only_correct
    mcnemar_p = (
        binomtest(a_only_correct, discordant, p=0.5, alternative='two-sided').pvalue
        if discordant else 1.0
    )

    pair_seed = BOOTSTRAP_SEED + 1009 * seed_a + 9176 * seed_b
    rng = np.random.default_rng(pair_seed)
    deltas = np.empty(BOOTSTRAP_REPLICATES, dtype=np.float64)
    n = len(a)
    chunk = 1000
    for start in range(0, BOOTSTRAP_REPLICATES, chunk):
        size = min(chunk, BOOTSTRAP_REPLICATES - start)
        indexes = rng.integers(0, n, size=(size, n))
        deltas[start:start + size] = (
            error_a[indexes].mean(axis=1) - error_b[indexes].mean(axis=1))
    low, high = np.quantile(deltas, [0.025, 0.975])
    return {
        'seed_a': seed_a,
        'seed_b': seed_b,
        'CI_a': error_a.mean(),
        'CI_b': error_b.mean(),
        'delta_CI_a_minus_b': error_a.mean() - error_b.mean(),
        'bootstrap_95_low': low,
        'bootstrap_95_high': high,
        'bootstrap_two_sided_p': min(1.0, 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())),
        'a_only_correct': a_only_correct,
        'b_only_correct': b_only_correct,
        'discordant': discordant,
        'mcnemar_exact_p': mcnemar_p,
    }


pairwise = pd.DataFrame([
    paired_comparison(seed_a, seed_b)
    for seed_a, seed_b in combinations(analysis_seeds, 2)
])
display(pairwise)
pairwise.to_csv(REPORT_DIR / 'paired_seed_tests.csv', index=False)


## 12. Robust seed and majority-vote ensemble


In [ ]:
pred_matrix = np.stack([
    prediction_frames[seed]['pred_idx'].to_numpy(dtype=int)
    for seed in analysis_seeds
])
gold = reference['gold_idx'].to_numpy(dtype=int)


ensemble_pred = []
for item_index in range(pred_matrix.shape[1]):
    votes = pred_matrix[:, item_index]
    counts = np.bincount(votes, minlength=3)
    winners = np.flatnonzero(counts == counts.max())
    if len(winners) == 1:
        ensemble_pred.append(int(winners[0]))
    else:
        ensemble_pred.append(int(pred_matrix[analysis_seeds.index(42), item_index]))
ensemble_pred = np.asarray(ensemble_pred)

agreement = np.zeros((len(analysis_seeds), len(analysis_seeds)))
for i in range(len(analysis_seeds)):
    for j in range(len(analysis_seeds)):
        agreement[i, j] = (pred_matrix[i] == pred_matrix[j]).mean()
mean_agreement = agreement.mean(axis=1)
medoid_seed = min(
    (analysis_seeds[i] for i in np.flatnonzero(mean_agreement == mean_agreement.max())),
)

ensemble = pd.DataFrame({
    'id': reference['id'],
    'gold_idx': gold,
    'pred_idx': ensemble_pred,
    'correct': ensemble_pred == gold,
})
ensemble.to_csv(REPORT_DIR / 'majority_ensemble_dev_predictions.csv', index=False)

ensemble_ci = (ensemble_pred != gold).mean()
print(f'Majority ensemble: accuracy={1-ensemble_ci:.4f}, CI={ensemble_ci:.4f}')
print(f'Seed medoid (highest mean prediction agreement): {medoid_seed}')
print('Mean agreement:', dict(zip(analysis_seeds, mean_agreement.round(4))))


## 13. Machine-readable final report and compact archive


In [ ]:
import zipfile

final_report = {
    'recipe': RECIPE_NAME,
    'analysis_seeds': analysis_seeds,
    'individual': individual.to_dict(orient='records'),
    'aggregate': aggregate.iloc[0].to_dict(),
    'pairwise_tests': pairwise.to_dict(orient='records'),
    'majority_ensemble_CI': float(ensemble_ci),
    'majority_ensemble_accuracy': float(1 - ensemble_ci),
    'medoid_seed': int(medoid_seed),
    'selection_note': (
        'Use the pre-registered majority ensemble for reporting. The medoid is a '
        'prediction-stability diagnostic, not a post-hoc best-dev model selection rule.'
    ),
}
with open(REPORT_DIR / 'final_report.json', 'w', encoding='utf-8') as handle:
    json.dump(final_report, handle, indent=2, ensure_ascii=False)

archive_path = OUTPUT_ROOT / 'experiment2_reports.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(REPORT_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_ROOT))
print('Reports:', REPORT_DIR)
print('Compact report archive:', archive_path)
print('Adapters remain in each seed directory and are intentionally excluded from the report archive.')
